# GRPO Pipeline for PostgreSQL Configuration Tuning
## Using Qwen2.5-7B with Cost Model as Reward

This notebook implements Group Relative Policy Optimization (GRPO) to tune PostgreSQL configurations based on workload features, query plans, and internal metrics.

## 1. Setup and Dependencies

In [ ]:
!pip install torch transformers trl accelerate pandas numpy scikit-learn joblib peft bitsandbytes

In [ ]:
import torch
import pandas as pd
import numpy as np
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from typing import Dict, List, Tuple, Any
import joblib
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load Data and Models

In [ ]:
# Load your dataset
df = pd.read_csv('your_dataset.csv')  # Replace with your actual file path
print(df.info())
print("\nFirst few rows:")
print(df.head())

In [ ]:
# Configuration parameters for PostgreSQL
POSTGRES_KNOBS = [
    "shared_buffers",
    "work_mem",
    "maintenance_work_mem",
    "effective_cache_size",
    "max_connections",
    "wal_buffers",
    "checkpoint_completion_target",
    "checkpoint_timeout",
    "effective_io_concurrency",
    "join_collapse_limit",
    "from_collapse_limit",
    "bgwriter_delay",
    "bgwriter_lru_multiplier",
    "default_statistics_target",
    "max_parallel_workers_per_gather"
]

METRIC_FEATURES = [
    "xact_commit", "xact_rollback", "blks_read", "blks_hit",
    "tup_returned", "tup_fetched", "tup_inserted", "conflicts",
    "tup_updated", "tup_deleted", "disk_read_count", "disk_write_count",
    "disk_read_bytes", "disk_write_bytes"
]

In [ ]:
# Load the cost model (reward model)
# Replace with your actual cost model path
cost_model = joblib.load('voting_regressor_model.pkl')
print("Cost model loaded successfully")

## 3. Data Processing and Feature Engineering

In [ ]:
class ConfigurationNormalizer:
    """Normalizer for PostgreSQL configurations"""
    
    def __init__(self):
        # Define reasonable ranges for each parameter (min, max)
        self.knob_ranges = {
            "shared_buffers": (128, 16384),  # MB
            "work_mem": (4, 1024),  # MB
            "maintenance_work_mem": (64, 2048),  # MB
            "effective_cache_size": (1024, 32768),  # MB
            "max_connections": (10, 500),
            "wal_buffers": (8, 256),  # MB
            "checkpoint_completion_target": (0.5, 0.9),
            "checkpoint_timeout": (300, 3600),  # seconds
            "effective_io_concurrency": (1, 200),
            "join_collapse_limit": (1, 20),
            "from_collapse_limit": (1, 20),
            "bgwriter_delay": (10, 10000),  # ms
            "bgwriter_lru_multiplier": (1.0, 10.0),
            "default_statistics_target": (10, 1000),
            "max_parallel_workers_per_gather": (0, 8)
        }
        
    def normalize_config(self, config: Dict[str, float]) -> np.ndarray:
        """Normalize configuration values to 0-1 range"""
        normalized = []
        for knob in POSTGRES_KNOBS:
            value = config.get(knob, 0)
            min_val, max_val = self.knob_ranges[knob]
            normalized_val = (value - min_val) / (max_val - min_val)
            normalized_val = np.clip(normalized_val, 0, 1)
            normalized.append(normalized_val)
        return np.array(normalized)
    
    def denormalize_config(self, normalized: np.ndarray) -> Dict[str, float]:
        """Convert normalized values back to actual configuration"""
        config = {}
        for i, knob in enumerate(POSTGRES_KNOBS):
            min_val, max_val = self.knob_ranges[knob]
            value = normalized[i] * (max_val - min_val) + min_val
            config[knob] = round(value, 2)
        return config

normalizer = ConfigurationNormalizer()

In [ ]:
def parse_json_column(df: pd.DataFrame, column: str) -> pd.DataFrame:
    """Parse JSON string columns into dictionaries"""
    def safe_json_parse(x):
        # Handle None, NaN, empty strings
        if pd.isna(x) or x == '' or x is None:
            return {}
        
        # If already a dict, return as-is
        if isinstance(x, dict):
            return x
        
        # Try parsing as string
        if isinstance(x, str):
            # First, try standard JSON parsing
            try:
                return json.loads(x)
            except (json.JSONDecodeError, ValueError):
                pass
            
            # If JSON fails, try Python literal_eval (for dict strings with single quotes)
            try:
                import ast
                result = ast.literal_eval(x)
                if isinstance(result, dict):
                    return result
            except (ValueError, SyntaxError):
                pass
            
            # Last resort: try replacing single quotes with double quotes
            try:
                fixed_json = x.replace("'", '"')
                return json.loads(fixed_json)
            except (json.JSONDecodeError, ValueError):
                pass
            
            # If all parsing attempts fail
            print(f"Warning: Failed to parse in column '{column}': {x[:80]}...")
            return {}
        
        # Fallback for unexpected types
        return {}
    
    df[column] = df[column].apply(safe_json_parse)
    return df

# Parse JSON columns with error handling
for col in ['workload_full', 'configuration', 'internal_metrics', 'workload_features', 'query_plans']:
    if col in df.columns:
        print(f"Parsing column: {col}")
        df = parse_json_column(df, col)
        
        # Check for empty/invalid entries
        invalid_count = df[col].apply(lambda x: len(x) == 0).sum()
        if invalid_count > 0:
            print(f"  Warning: {invalid_count} rows have empty/invalid JSON in '{col}'")

print("\nData parsed successfully")
print(f"Total rows: {len(df)}")

# Optional: Drop rows where all JSON columns are empty
json_cols = ['workload_full', 'configuration', 'internal_metrics', 'workload_features', 'query_plans']
before_len = len(df)

# Drop rows where ALL json columns are empty
df = df[~df[json_cols].apply(lambda row: all(len(row[col]) == 0 for col in json_cols if col in df.columns), axis=1)]

after_len = len(df)
if before_len > after_len:
    print(f"\nDropped {before_len - after_len} rows with all empty JSON columns")
    print(f"Remaining rows: {after_len}")

## 4. Prompt Engineering for Configuration Generation

In [ ]:
def create_input_prompt(workload_features: Dict, internal_metrics: Dict, query_plans: List) -> str:
    """Create input prompt for the LLM"""
    
    # Handle query_plans as a list
    num_queries = len(query_plans) if isinstance(query_plans, list) else 0
    
    # Calculate complexity based on query count (simple heuristic)
    if num_queries == 0:
        complexity = "Unknown"
    elif num_queries < 5:
        complexity = "Low"
    elif num_queries < 15:
        complexity = "Medium"
    else:
        complexity = "High"
    
    # Optionally summarize query plans
    query_summary = ""
    if num_queries > 0 and isinstance(query_plans, list):
        # Sample first few queries for context
        sample_queries = query_plans[:3] if len(query_plans) > 3 else query_plans
        query_summary = f"\nSample queries:\n{json.dumps(sample_queries, indent=2)}"
    
    prompt = f"""You are a PostgreSQL database configuration expert. Generate optimal configuration values for the following workload.

### Workload Features:
{json.dumps(workload_features, indent=2)}

### Internal Metrics:
{json.dumps(internal_metrics, indent=2)}

### Query Plan Summary:
Number of queries: {num_queries}
Complexity: {complexity}{query_summary}

### Task:
Generate optimal PostgreSQL configuration values for the following parameters. Provide ONLY numeric values in JSON format.

Required parameters:
- shared_buffers (MB): Memory for shared buffer pool (128-16384)
- work_mem (MB): Memory per query operation (4-1024)
- maintenance_work_mem (MB): Memory for maintenance operations (64-2048)
- effective_cache_size (MB): Planner's assumption of cache size (1024-32768)
- max_connections: Maximum concurrent connections (10-500)
- wal_buffers (MB): WAL buffer size (8-256)
- checkpoint_completion_target: Target for checkpoint completion (0.5-0.9)
- checkpoint_timeout (seconds): Time between checkpoints (300-3600)
- effective_io_concurrency: Concurrent disk I/O operations (1-200)
- join_collapse_limit: Joins to reorder with GEQO (1-20)
- from_collapse_limit: FROM items to reorder with GEQO (1-20)
- bgwriter_delay (ms): Background writer delay (10-10000)
- bgwriter_lru_multiplier: LRU multiplier (1.0-10.0)
- default_statistics_target: Statistics target (10-1000)
- max_parallel_workers_per_gather: Parallel workers per gather (0-8)

### Output Format (JSON only):
{{
  "shared_buffers": <value>,
  "work_mem": <value>,
  "maintenance_work_mem": <value>,
  "effective_cache_size": <value>,
  "max_connections": <value>,
  "wal_buffers": <value>,
  "checkpoint_completion_target": <value>,
  "checkpoint_timeout": <value>,
  "effective_io_concurrency": <value>,
  "join_collapse_limit": <value>,
  "from_collapse_limit": <value>,
  "bgwriter_delay": <value>,
  "bgwriter_lru_multiplier": <value>,
  "default_statistics_target": <value>,
  "max_parallel_workers_per_gather": <value>
}}

Configuration:"""
    
    return prompt

# Test prompt creation
if len(df) > 0:
    sample_row = df.iloc[0]
    test_prompt = create_input_prompt(
        sample_row['workload_features'],
        sample_row['internal_metrics'],
        sample_row['query_plans']
    )
    print("Sample prompt created:")
    print(test_prompt[:500] + "...")
else:
    print("ERROR: DataFrame is empty. Check data loading and parsing.")

## 5. Load and Prepare the LLM

In [ ]:
# Hugging Face token - replace with your token
hf_token = "your_huggingface_token_here"

model_id = "Qwen/Qwen2.5-7B"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=hf_token,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=hf_token,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded successfully")

In [ ]:
# Configure LoRA for parameter-efficient fine-tuning
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Reward Model (Cost Model Wrapper)

In [ ]:
class RewardModel:
    """Wrapper for the cost model to use as reward function"""
    
    def __init__(self, cost_model, normalizer):
        self.cost_model = cost_model
        self.normalizer = normalizer
        
    def prepare_features(self, config: Dict[str, float], metrics: Dict[str, float]) -> np.ndarray:
        """Prepare features for the cost model"""
        # Normalize configuration
        cfg_features = self.normalizer.normalize_config(config)
        
        # Normalize metrics
        metric_values = []
        for metric in METRIC_FEATURES:
            value = metrics.get(metric, 0)
            metric_values.append(value)
        
        # Combine features
        all_features = np.concatenate([cfg_features, metric_values])
        
        return all_features.reshape(1, -1)
    
    def calculate_reward(self, config: Dict[str, float], metrics: Dict[str, float]) -> float:
        """Calculate reward (higher TPS = higher reward)"""
        try:
            features = self.prepare_features(config, metrics)
            
            # Predict cost (lower is better)
            predicted_cost = self.cost_model.predict(features)[0]
            
            # Convert to reward (inverse of cost, scaled)
            # Higher TPS = lower cost = higher reward
            reward = 1.0 / (predicted_cost + 1e-6)
            
            # Normalize reward to reasonable range
            reward = np.clip(reward * 0.1, -10, 10)
            
            return float(reward)
        except Exception as e:
            print(f"Error calculating reward: {e}")
            return -1.0

reward_model = RewardModel(cost_model, normalizer)
print("Reward model initialized")

## 7. Configuration Generator and Parser

In [ ]:
def parse_llm_output(text: str) -> Dict[str, float]:
    """Parse LLM output to extract configuration"""
    try:
        # Try to find JSON in the output
        start_idx = text.find('{')
        end_idx = text.rfind('}') + 1
        
        if start_idx != -1 and end_idx > start_idx:
            json_str = text[start_idx:end_idx]
            config = json.loads(json_str)
            
            # Validate all required keys are present
            for knob in POSTGRES_KNOBS:
                if knob not in config:
                    raise ValueError(f"Missing knob: {knob}")
            
            return config
        else:
            raise ValueError("No JSON found in output")
    except Exception as e:
        print(f"Error parsing output: {e}")
        # Return default configuration
        return {
            "shared_buffers": 2048,
            "work_mem": 64,
            "maintenance_work_mem": 256,
            "effective_cache_size": 8192,
            "max_connections": 100,
            "wal_buffers": 16,
            "checkpoint_completion_target": 0.7,
            "checkpoint_timeout": 900,
            "effective_io_concurrency": 100,
            "join_collapse_limit": 8,
            "from_collapse_limit": 8,
            "bgwriter_delay": 200,
            "bgwriter_lru_multiplier": 2.0,
            "default_statistics_target": 100,
            "max_parallel_workers_per_gather": 2
        }

def generate_configuration(model, tokenizer, prompt: str, max_new_tokens: int = 512) -> str:
    """Generate configuration using the model"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the generated part
    generated_config = generated_text[len(prompt):]
    
    return generated_config

## 8. GRPO Dataset Preparation

In [ ]:
from torch.utils.data import Dataset

class PostgresConfigDataset(Dataset):
    """Dataset for GRPO training"""
    
    def __init__(self, df, tokenizer):
        self.df = df
        self.tokenizer = tokenizer
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Create prompt
        prompt = create_input_prompt(
            row['workload_features'],
            row['internal_metrics'],
            row['query_plans']
        )
        
        # Store additional data for reward calculation
        return {
            'prompt': prompt,
            'metrics': row['internal_metrics'],
            'baseline_performance': row.get('default_config_performance', 0)
        }

# Create dataset
train_dataset = PostgresConfigDataset(df, tokenizer)
print(f"Dataset size: {len(train_dataset)}")

## 9. Custom GRPO Training Loop

In [ ]:
class GRPOTrainerCustom:
    """Custom GRPO trainer for PostgreSQL configuration tuning"""
    
    def __init__(self, model, tokenizer, reward_model, train_dataset, 
                 num_epochs=5, batch_size=4, learning_rate=5e-5):
        self.model = model
        self.tokenizer = tokenizer
        self.reward_model = reward_model
        self.train_dataset = train_dataset
        self.num_epochs = num_epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
        self.history = []
        
    def generate_group_responses(self, prompts: List[str], group_size: int = 4) -> List[Tuple[str, Dict]]:
        """Generate multiple responses for each prompt (GRPO groups)"""
        all_responses = []
        
        for prompt in prompts:
            group_responses = []
            for _ in range(group_size):
                response = generate_configuration(self.model, self.tokenizer, prompt)
                config = parse_llm_output(response)
                group_responses.append((response, config))
            all_responses.append(group_responses)
        
        return all_responses
    
    def calculate_group_rewards(self, group_responses: List[Tuple[str, Dict]], 
                                metrics: Dict) -> List[float]:
        """Calculate rewards for a group of responses"""
        rewards = []
        for _, config in group_responses:
            reward = self.reward_model.calculate_reward(config, metrics)
            rewards.append(reward)
        return rewards
    
    def compute_grpo_loss(self, responses: List[str], rewards: List[float]) -> torch.Tensor:
        """Compute GRPO loss"""
        # Normalize rewards (group normalization)
        rewards = np.array(rewards)
        normalized_rewards = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
        
        total_loss = 0
        
        for response, reward in zip(responses, normalized_rewards):
            # Tokenize response
            inputs = self.tokenizer(response, return_tensors="pt", truncation=True).to(self.model.device)
            
            # Get model outputs
            outputs = self.model(**inputs, labels=inputs["input_ids"])
            
            # Weighted loss by reward (higher reward = lower weight for loss)
            loss = outputs.loss * (-reward)  # Negative because we want to maximize reward
            total_loss += loss
        
        return total_loss / len(responses)
    
    def train_epoch(self, epoch: int):
        """Train for one epoch"""
        self.model.train()
        epoch_rewards = []
        
        for batch_start in range(0, len(self.train_dataset), self.batch_size):
            batch_end = min(batch_start + self.batch_size, len(self.train_dataset))
            batch_indices = range(batch_start, batch_end)
            
            # Get batch data
            batch_data = [self.train_dataset[i] for i in batch_indices]
            prompts = [item['prompt'] for item in batch_data]
            metrics_list = [item['metrics'] for item in batch_data]
            
            # Generate group responses
            all_group_responses = self.generate_group_responses(prompts, group_size=4)
            
            # Calculate rewards for each group
            batch_loss = 0
            for group_responses, metrics in zip(all_group_responses, metrics_list):
                rewards = self.calculate_group_rewards(group_responses, metrics)
                epoch_rewards.extend(rewards)
                
                # Extract response texts
                response_texts = [resp for resp, _ in group_responses]
                
                # Compute GRPO loss
                loss = self.compute_grpo_loss(response_texts, rewards)
                batch_loss += loss
            
            # Optimize
            self.optimizer.zero_grad()
            batch_loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optimizer.step()
            
            # Log progress
            if batch_start % 10 == 0:
                avg_reward = np.mean(epoch_rewards[-len(rewards):])
                print(f"Epoch {epoch}, Batch {batch_start//self.batch_size}, "
                      f"Loss: {batch_loss.item():.4f}, Avg Reward: {avg_reward:.4f}")
        
        return np.mean(epoch_rewards)
    
    def train(self):
        """Full training loop"""
        print("Starting GRPO training...")
        
        for epoch in range(self.num_epochs):
            print(f"\n{'='*60}")
            print(f"Epoch {epoch + 1}/{self.num_epochs}")
            print(f"{'='*60}")
            
            avg_reward = self.train_epoch(epoch)
            self.history.append(avg_reward)
            
            print(f"\nEpoch {epoch + 1} completed. Average Reward: {avg_reward:.4f}")
            
            # Save checkpoint
            if (epoch + 1) % 2 == 0:
                self.save_checkpoint(f"checkpoint_epoch_{epoch+1}")
        
        print("\nTraining completed!")
        return self.history
    
    def save_checkpoint(self, name: str):
        """Save model checkpoint"""
        self.model.save_pretrained(f"./{name}")
        self.tokenizer.save_pretrained(f"./{name}")
        print(f"Checkpoint saved: {name}")

## 10. Train the Model

In [ ]:
# Initialize trainer
trainer = GRPOTrainerCustom(
    model=model,
    tokenizer=tokenizer,
    reward_model=reward_model,
    train_dataset=train_dataset,
    num_epochs=5,
    batch_size=2,  # Adjust based on GPU memory
    learning_rate=5e-5
)

# Start training
history = trainer.train()

## 11. Visualize Training Progress

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(history, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Average Reward')
plt.title('GRPO Training Progress')
plt.grid(True)
plt.savefig('training_progress.png')
plt.show()

print(f"\nFinal average reward: {history[-1]:.4f}")
print(f"Improvement: {(history[-1] - history[0]):.4f}")

## 12. Evaluate and Generate Configurations

In [ ]:
def evaluate_model(model, tokenizer, reward_model, test_data, num_samples=10):
    """Evaluate model on test data"""
    model.eval()
    results = []
    
    for i in range(min(num_samples, len(test_data))):
        row = test_data.iloc[i]
        
        # Create prompt
        prompt = create_input_prompt(
            row['workload_features'],
            row['internal_metrics'],
            row['query_plans']
        )
        
        # Generate configuration
        response = generate_configuration(model, tokenizer, prompt)
        config = parse_llm_output(response)
        
        # Calculate reward
        reward = reward_model.calculate_reward(config, row['internal_metrics'])
        
        results.append({
            'sample_id': i,
            'generated_config': config,
            'reward': reward,
            'baseline_performance': row.get('default_config_performance', 0)
        })
        
        print(f"\nSample {i+1}:")
        print(f"Generated Config: {json.dumps(config, indent=2)}")
        print(f"Reward: {reward:.4f}")
        print(f"Baseline Performance: {row.get('default_config_performance', 0):.4f}")
    
    return results

# Evaluate on test set
test_results = evaluate_model(model, tokenizer, reward_model, df, num_samples=5)

## 13. Save Final Model and Results

In [ ]:
# Save final model
model.save_pretrained("./postgres_grpo_final")
tokenizer.save_pretrained("./postgres_grpo_final")

# Save results
results_df = pd.DataFrame(test_results)
results_df.to_csv('grpo_evaluation_results.csv', index=False)

print("Model and results saved successfully!")

## 14. Inference Pipeline

In [ ]:
def inference_pipeline(workload_features, internal_metrics, query_plans, 
                       model, tokenizer, num_candidates=5):
    """
    Complete inference pipeline:
    1. Generate multiple candidate configurations
    2. Rank by predicted reward
    3. Return best configuration
    """
    model.eval()
    
    # Create prompt
    prompt = create_input_prompt(workload_features, internal_metrics, query_plans)
    
    # Generate multiple candidates
    candidates = []
    for i in range(num_candidates):
        response = generate_configuration(model, tokenizer, prompt)
        config = parse_llm_output(response)
        reward = reward_model.calculate_reward(config, internal_metrics)
        candidates.append((config, reward))
    
    # Sort by reward
    candidates.sort(key=lambda x: x[1], reverse=True)
    
    print("\n" + "="*60)
    print("TOP CONFIGURATIONS")
    print("="*60)
    
    for i, (config, reward) in enumerate(candidates[:3]):
        print(f"\nRank {i+1} (Reward: {reward:.4f}):")
        print(json.dumps(config, indent=2))
    
    return candidates[0][0], candidates[0][1]

# Example usage
sample = df.iloc[0]
best_config, best_reward = inference_pipeline(
    sample['workload_features'],
    sample['internal_metrics'],
    sample['query_plans'],
    model,
    tokenizer
)

## 15. Export Production-Ready Model

In [ ]:
# Merge LoRA weights back into base model for production
from peft import PeftModel

merged_model = model.merge_and_unload()
merged_model.save_pretrained("./postgres_grpo_production")
tokenizer.save_pretrained("./postgres_grpo_production")

print("Production model saved!")
print("\nTo load in production:")
print("model = AutoModelForCausalLM.from_pretrained('./postgres_grpo_production')")
print("tokenizer = AutoTokenizer.from_pretrained('./postgres_grpo_production')")